In [1]:
import os
os.environ["VISIBLE_DEVICES"] = "0"
import onnxruntime as ort
import torchaudio
import torch
import numpy as np

In [3]:
# ONNXロード
MODEL_PATH = "./speech_tokenizer_v2.onnx"
session = ort.InferenceSession(
    MODEL_PATH,
    providers=["CPUExecutionProvider"]
)

In [4]:
# 音声ロード
SPEECH_PATH = "/db/Coco-Nut/Coco-Nut/wav/0/0000.wav"
wav, sr = torchaudio.load(SPEECH_PATH)

# 16kHzにリサンプル
if sr != 16000:
    wav = torchaudio.functional.resample(wav, sr, 16000)
# モノラル化
wav = wav.mean(dim=0, keepdim=True)  # [1, T]

In [5]:
# log-mel spectrogram作成
mel_transform = torchaudio.transforms.MelSpectrogram(
    sample_rate=16000,
    n_fft=400,         # 25ms
    hop_length=320,    # 10ms=>160 samples, 20ms=>320 samples
    win_length=400,
    n_mels=128,
    f_min=0,
    f_max=8000,
    center=True,
    power=2.0
)

mel = mel_transform(wav)            # [1, 128, T]
log_mel = torch.log(mel + 1e-6)     # [1, 128, T]


# length作成
# データ型は int32 で、フレーム数は T = wav_length // (hop_length = 160)
feats = log_mel.numpy().astype(np.float32)
feats_length = np.array([feats.shape[1]], dtype=np.int32)

multiple = 32
T = feats.shape[2]
pad_len = (multiple - (T % multiple)) % multiple

if pad_len > 0:
    pad = np.zeros((feats.shape[0], feats.shape[1], pad_len), dtype=feats.dtype)
    feats = np.concatenate([feats, pad], axis=2)

/workH/isa/python/prj_cosy/.venv/lib/python3.12/site-packages/torchaudio/functional/functional.py:582: UserWarning: At least one mel filterbank has all zero values. The value for `n_mels` (128) may be set too high. Or, the value for `n_freqs` (201) may be set too low.
  warnings.warn(


In [6]:
# 推論
# for i in session.get_inputs():
#     print("Input:", i.name, i.shape, i.type)

outputs = session.run(
    None,
    {
        "feats": feats,
        "feats_length": feats_length
    }
)

tokens = outputs[0]

print("speech tokens shape:", tokens.shape)
print(tokens)

speech tokens shape: (1, 32)
[[3329 5352 4624 1536 1510 3211 3183  763 1482 1725 4623 1500 1537  780
  1725 1473  745 2958 1509 1725 1707 2931 3696 1482 3669 1509 1482 2958
  1789 3976 6405 6405]]
